# Bayesian Stats Snippets

A small collection of Bayesian probability examples:

- conditional probabilities from a contingency table
- a binary Bayesian update helper
- sequential updates from multiple pieces of evidence
- the same update written in odds and log-odds form

These are meant to be readable reference snippets rather than a full statistics lesson.

In [ ]:
import numpy as np

np.set_printoptions(precision=3, suppress=True)

## Conditional Probability From a Table

Titanic survival counts by passenger class / crew group:

| Outcome | 1st | 2nd | 3rd | Crew |
| --- | ---: | ---: | ---: | ---: |
| Survived | 203 | 118 | 178 | 212 |
| Did not survive | 122 | 167 | 528 | 673 |

Two easy-to-confuse questions:

- `P(Survived | First Class)`: among first-class passengers, how many survived?
- `P(First Class | Survived)`: among survivors, how many were first class?

In [ ]:
classes = np.array(["1st", "2nd", "3rd", "Crew"])
titanic = np.array([
    [203, 118, 178, 212],  # survived
    [122, 167, 528, 673],  # did not survive
], dtype=float)

survived = titanic[0]
not_survived = titanic[1]
class_totals = titanic.sum(axis=0)
total = titanic.sum()

overall_survival_rate = survived.sum() / total
survival_rates_by_class = survived / class_totals

print(f"Total passengers/crew: {int(total)}")
print(f"Overall survival rate: {overall_survival_rate:.1%}")
print()

for label, group_total, survival_rate in zip(
    classes,
    class_totals,
    survival_rates_by_class,
):
    print(f"{label:>4}: {int(group_total):>4} people, {survival_rate:>6.1%} survived")

In [ ]:
first_class = 0

p_first = class_totals[first_class] / total
p_survived = survived.sum() / total
p_first_and_survived = survived[first_class] / total

p_survived_given_first = p_first_and_survived / p_first
p_first_given_survived = p_first_and_survived / p_survived

print(f"P(First Class):             {p_first:.3f}")
print(f"P(Survived):                {p_survived:.3f}")
print(f"P(First Class and Survived): {p_first_and_survived:.3f}")
print()
print(f"P(Survived | First Class):  {p_survived_given_first:.3f}")
print(f"P(First Class | Survived):  {p_first_given_survived:.3f}")

## A Binary Bayesian Update

For a hypothesis `H` and observed evidence `E`:

```text
posterior = P(H | E)
          = P(E | H) P(H) / P(E)

P(E) = P(E | H) P(H) + P(E | not H) P(not H)
```

In code, that means the posterior is the normalized signal:

```text
signal = P(E | H) * P(H)
noise  = P(E | not H) * P(not H)
posterior = signal / (signal + noise)
```

In [ ]:
def require_probability(value: float, name: str) -> float:
    """Return value as a float after checking that it is a probability."""
    value = float(value)
    if not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be between 0 and 1; got {value}")
    return value


def bayes_update(likelihood_h1: float, likelihood_h0: float, prior: float) -> float:
    """
    Update P(H=1) after observing evidence E.

    likelihood_h1 is P(E | H=1).
    likelihood_h0 is P(E | H=0).
    prior is P(H=1) before observing E.
    """
    likelihood_h1 = require_probability(likelihood_h1, "likelihood_h1")
    likelihood_h0 = require_probability(likelihood_h0, "likelihood_h0")
    prior = require_probability(prior, "prior")

    signal = likelihood_h1 * prior
    noise = likelihood_h0 * (1.0 - prior)
    evidence_probability = signal + noise

    if evidence_probability == 0.0:
        raise ValueError("Evidence has zero probability under both hypotheses.")

    return signal / evidence_probability


def apply_bayes_updates(prior: float, evidence):
    """Apply a sequence of (label, P(E|H=1), P(E|H=0)) observations."""
    belief = require_probability(prior, "prior")
    history = [("Prior", belief)]

    for label, likelihood_h1, likelihood_h0 in evidence:
        belief = bayes_update(likelihood_h1, likelihood_h0, belief)
        history.append((label, belief))

    return history

In [ ]:
prior = 0.40
sensor_evidence = [
    ("Sensor A", 0.75, 0.25),
    ("Sensor B", 0.60, 0.20),
]

for label, belief in apply_bayes_updates(prior, sensor_evidence):
    print(f"{label:>8}: P(H=1) = {belief:.3f}")

## Odds and Log-Odds

Bayesian updates can also be written in odds form. This is often the most natural way to think about Bayes because both the prior belief and the new evidence are comparisons.

Odds compare belief for a hypothesis against belief against it:

```text
odds = P(H=1) / P(H=0)
```

So if the odds are `3`, the hypothesis is three times as plausible as its alternative. Evidence has the same shape: it asks how much more likely the observation would be if the hypothesis were true than if it were false.

That evidence weight is the likelihood ratio:

```text
posterior odds = prior odds * likelihood ratio
likelihood ratio = P(E | H=1) / P(E | H=0)
```

This makes Bayes feel like a belief accumulator: start with your prior odds, then multiply by each new piece of evidence.

Taking logs turns that multiplication into addition:

```text
posterior log-odds = prior log-odds + log likelihood ratio
```

This is useful because each observation becomes an additive push:

- positive log likelihood ratio: evidence pushes toward `H=1`
- negative log likelihood ratio: evidence pushes away from `H=1`
- zero log likelihood ratio: evidence is neutral

That is why log-odds shows up in logistic regression, Bayesian filtering, and sensor fusion. The model can keep a running score, add evidence as it arrives, and convert back to probability only when it needs a human-readable answer.

In [ ]:
def probability_to_odds(probability: float) -> float:
    probability = require_probability(probability, "probability")
    if probability in (0.0, 1.0):
        raise ValueError("Odds are infinite when probability is exactly 0 or 1.")
    return probability / (1.0 - probability)


def odds_to_probability(odds: float) -> float:
    odds = float(odds)
    if odds < 0.0:
        raise ValueError(f"odds must be non-negative; got {odds}")
    return odds / (1.0 + odds)


def probability_to_log_odds(probability: float) -> float:
    return np.log(probability_to_odds(probability))


def log_odds_to_probability(log_odds: float) -> float:
    return 1.0 / (1.0 + np.exp(-float(log_odds)))


def update_log_odds(
    log_likelihood_h1: float,
    log_likelihood_h0: float,
    prior_log_odds: float = 0.0,
) -> float:
    """Update log-odds using log(P(E|H=1)) and log(P(E|H=0))."""
    log_likelihood_ratio = log_likelihood_h1 - log_likelihood_h0
    return prior_log_odds + log_likelihood_ratio


def update_log_odds_from_likelihoods(
    likelihood_h1: float,
    likelihood_h0: float,
    prior_log_odds: float = 0.0,
) -> float:
    """Update log-odds using regular probabilities."""
    likelihood_h1 = require_probability(likelihood_h1, "likelihood_h1")
    likelihood_h0 = require_probability(likelihood_h0, "likelihood_h0")
    if likelihood_h1 == 0.0 or likelihood_h0 == 0.0:
        raise ValueError("Log-odds updates need non-zero likelihoods.")

    return update_log_odds(
        np.log(likelihood_h1),
        np.log(likelihood_h0),
        prior_log_odds,
    )

In [ ]:
log_belief = probability_to_log_odds(prior)
print(f"{'Prior':>8}: log-odds = {log_belief: .3f}, P(H=1) = {log_odds_to_probability(log_belief):.3f}")

for label, likelihood_h1, likelihood_h0 in sensor_evidence:
    log_belief = update_log_odds_from_likelihoods(
        likelihood_h1,
        likelihood_h0,
        log_belief,
    )
    belief = log_odds_to_probability(log_belief)
    print(f"{label:>8}: log-odds = {log_belief: .3f}, P(H=1) = {belief:.3f}")

In [ ]:
probability_belief = prior
log_belief = probability_to_log_odds(prior)

for _, likelihood_h1, likelihood_h0 in sensor_evidence:
    probability_belief = bayes_update(likelihood_h1, likelihood_h0, probability_belief)
    log_belief = update_log_odds_from_likelihoods(
        likelihood_h1,
        likelihood_h0,
        log_belief,
    )

log_odds_belief = log_odds_to_probability(log_belief)
np.testing.assert_allclose(probability_belief, log_odds_belief)

print(f"Probability-space result: {probability_belief:.6f}")
print(f"Log-odds-space result:   {log_odds_belief:.6f}")

## Takeaways

- Conditional probabilities are directional: `P(A | B)` and `P(B | A)` usually differ.
- A Bayesian update combines a prior with likelihoods for the new evidence.
- Sequential updates reuse the posterior from one step as the prior for the next step.
- Odds/log-odds make repeated evidence updates additive, which is often cleaner for code.